# Chapter 12 Lab: Knowledge Graphs

```{admonition} Lab Objectives
:class: tip
- Build knowledge graphs with RDF
- Implement graph embeddings
- Perform link prediction
- Query with SPARQL
- Apply to real-world KGs
```

## Lab Overview
1. Knowledge graph construction
2. TransE and other embeddings
3. Link prediction
4. Knowledge completion
5. Real-world applications

## Exercise 1: Building Knowledge Graphs

Create and manipulate knowledge graphs.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from typing import Tuple, List

class KnowledgeGraph:
    def __init__(self):
        self.graph = nx.MultiDiGraph()
        self.entities = set()
        self.relations = set()
        
    def add_triple(self, head: str, relation: str, tail: str):
        """Add (head, relation, tail) triple"""
        self.graph.add_edge(head, tail, relation=relation)
        self.entities.add(head)
        self.entities.add(tail)
        self.relations.add(relation)
        
    def get_neighbors(self, entity: str, relation: str = None) -> List[str]:
        """
        TODO: Get neighbors of entity
        If relation specified, filter by relation type
        """
        # YOUR CODE HERE
        pass
        
    def get_paths(self, start: str, end: str, max_length: int = 3) -> List[List[str]]:
        """
        TODO: Find all paths between entities
        Return paths as list of (entity, relation) tuples
        """
        # YOUR CODE HERE
        pass
        
    def visualize(self, figsize=(12, 8)):
        plt.figure(figsize=figsize)
        pos = nx.spring_layout(self.graph, k=2, iterations=50)
        
        # Draw nodes
        nx.draw_networkx_nodes(self.graph, pos, node_size=500, node_color='lightblue')
        nx.draw_networkx_labels(self.graph, pos)
        
        # Draw edges with labels
        for (u, v, key, data) in self.graph.edges(keys=True, data=True):
            relation = data.get('relation', '')
            nx.draw_networkx_edges(self.graph, pos, [(u, v)], 
                                  connectionstyle='arc3,rad=0.1')
            # Add edge labels
            edge_pos = {(u, v): ((pos[u][0] + pos[v][0])/2, 
                                 (pos[u][1] + pos[v][1])/2)}
            nx.draw_networkx_edge_labels(self.graph, pos, 
                                        {(u, v): relation})
        plt.axis('off')
        plt.show()

# Build example KG
kg = KnowledgeGraph()

# Add movie knowledge
kg.add_triple('Inception', 'directed_by', 'Christopher Nolan')
kg.add_triple('Inception', 'starring', 'Leonardo DiCaprio')
kg.add_triple('Inception', 'genre', 'Sci-Fi')
kg.add_triple('Christopher Nolan', 'directed', 'Interstellar')
kg.add_triple('Interstellar', 'starring', 'Matthew McConaughey')
kg.add_triple('Leonardo DiCaprio', 'acted_in', 'Titanic')
kg.add_triple('Titanic', 'directed_by', 'James Cameron')

kg.visualize()

# Query
print('Movies directed by Christopher Nolan:')
print(kg.get_neighbors('Christopher Nolan', 'directed'))

## Exercise 2: Graph Embeddings - TransE

Implement TransE embedding model.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class TransE(nn.Module):
    def __init__(self, n_entities, n_relations, embedding_dim=50, margin=1.0):
        super(TransE, self).__init__()
        self.embedding_dim = embedding_dim
        self.margin = margin
        
        # Embeddings
        self.entity_embeddings = nn.Embedding(n_entities, embedding_dim)
        self.relation_embeddings = nn.Embedding(n_relations, embedding_dim)
        
        # Initialize
        nn.init.uniform_(self.entity_embeddings.weight, -6/np.sqrt(embedding_dim), 
                        6/np.sqrt(embedding_dim))
        nn.init.uniform_(self.relation_embeddings.weight, -6/np.sqrt(embedding_dim), 
                        6/np.sqrt(embedding_dim))
        
    def forward(self, heads, relations, tails):
        """
        TODO: Compute TransE score
        
        TransE: h + r ≈ t
        Score: ||h + r - t||
        """
        # YOUR CODE HERE
        pass
        
    def loss(self, pos_triples, neg_triples):
        """
        TODO: Compute margin ranking loss
        
        L = max(0, γ + d(h+r, t) - d(h'+r, t'))
        where (h',r,t') is negative sample
        """
        # YOUR CODE HERE
        pass

def generate_negative_samples(triples, n_entities, n_samples=1):
    """
    TODO: Generate negative samples
    Randomly corrupt head or tail entity
    """
    # YOUR CODE HERE
    pass

# Prepare data
triples = []
entity2id = {}
relation2id = {}

for head, relation, tail in kg.graph.edges(data='relation'):
    if head not in entity2id:
        entity2id[head] = len(entity2id)
    if tail not in entity2id:
        entity2id[tail] = len(entity2id)
    if relation not in relation2id:
        relation2id[relation] = len(relation2id)
    
    triples.append((entity2id[head], relation2id[relation], entity2id[tail]))

triples = torch.LongTensor(triples)

# Train TransE
model = TransE(len(entity2id), len(relation2id), embedding_dim=50)
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    neg_triples = generate_negative_samples(triples, len(entity2id))
    
    optimizer.zero_grad()
    loss = model.loss(triples, neg_triples)
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

print('Training complete!')

## Exercise 3: Link Prediction

Predict missing links in KG.

In [ ]:
def predict_tail(model, head, relation, entity2id, top_k=5):
    """
    TODO: Predict most likely tail entities
    
    Compute h + r and find nearest entity embeddings
    """
    # YOUR CODE HERE
    pass

def link_prediction_evaluation(model, test_triples, entity2id):
    """
    TODO: Evaluate link prediction
    
    Metrics:
    - Mean Rank (MR)
    - Mean Reciprocal Rank (MRR)
    - Hits@k
    """
    # YOUR CODE HERE
    pass

# Test predictions
test_queries = [
    ('Inception', 'starring', '?'),
    ('Christopher Nolan', 'directed', '?')
]

for head, relation, _ in test_queries:
    head_id = entity2id[head]
    rel_id = relation2id[relation]
    predictions = predict_tail(model, head_id, rel_id, entity2id)
    print(f'{head} --{relation}--> {predictions}')

## Exercise 4: SPARQL Queries

Query knowledge graphs with SPARQL.

In [ ]:
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS

def create_rdf_graph(kg: KnowledgeGraph) -> Graph:
    """
    TODO: Convert KG to RDF graph
    Create proper URIs and add triples
    """
    g = Graph()
    ns = Namespace('http://example.org/')
    
    # YOUR CODE HERE
    pass
    
    return g

def execute_sparql(g: Graph, query: str):
    """Execute SPARQL query"""
    results = g.query(query)
    return list(results)

# Create RDF graph
rdf_graph = create_rdf_graph(kg)

# Example SPARQL queries
queries = {
    'All movies': """
        SELECT ?movie WHERE {
            ?movie rdf:type :Movie .
        }
    """,
    
    'Directors and their movies': """
        SELECT ?director ?movie WHERE {
            ?movie :directed_by ?director .
        }
    """,
    
    'Movies with common actors': """
        SELECT ?movie1 ?movie2 ?actor WHERE {
            ?movie1 :starring ?actor .
            ?movie2 :starring ?actor .
            FILTER(?movie1 != ?movie2)
        }
    """
}

for name, query in queries.items():
    print(f'\n{name}:')
    results = execute_sparql(rdf_graph, query)
    for row in results:
        print(row)

## Exercise 5: Knowledge Graph Completion

Implement rule-based reasoning.

In [ ]:
class Rule:
    def __init__(self, body: List[Tuple], head: Tuple):
        """
        Rule: body => head
        Example: (X, spouse, Y) ∧ (Y, parent, Z) => (X, step_parent, Z)
        """
        self.body = body
        self.head = head
        
    def apply(self, kg: KnowledgeGraph) -> List[Tuple]:
        """
        TODO: Apply rule to find new triples
        Use pattern matching on KG
        """
        # YOUR CODE HERE
        pass

def rule_mining(kg: KnowledgeGraph, min_support=2) -> List[Rule]:
    """
    TODO: Mine rules from KG
    Find frequent patterns that can be rules
    """
    # YOUR CODE HERE
    pass

# Define rules
rules = [
    Rule(
        body=[('?x', 'directed', '?y')],
        head=('?y', 'directed_by', '?x')
    ),
    Rule(
        body=[('?x', 'directed', '?y'), ('?y', 'starring', '?z')],
        head=('?x', 'worked_with', '?z')
    )
]

# Apply rules
for rule in rules:
    new_triples = rule.apply(kg)
    print(f'Rule produced {len(new_triples)} new triples')
    for triple in new_triples[:5]:
        print(f'  {triple}')

## Challenge: Multi-hop Reasoning

Implement path-based reasoning.

In [ ]:
def path_ranking(kg: KnowledgeGraph, query: Tuple[str, str, str], 
                 max_path_length=3) -> List[Tuple[List, float]]:
    """
    CHALLENGE: Rank paths for query answering
    
    Find paths from head to tail
    Score paths using path features
    Combine evidence from multiple paths
    """
    # YOUR CODE HERE
    pass

## Lab Report

### Deliverables
- [ ] Knowledge graph construction
- [ ] TransE implementation
- [ ] Link prediction
- [ ] SPARQL queries
- [ ] Rule-based completion

### Discussion
1. Compare different KG embedding methods
2. How do you evaluate link prediction quality?
3. What are advantages of symbolic vs neural KG methods?
4. Applications of knowledge graphs in industry